# 7교시. 추출 결과 검증 및 데이터 저장

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/07_validation_export.ipynb)

**이번 교시 행동:** 오류·경고·사람 검토를 분리하고, 공개된 승인 정답 경로에서만 Excel을 만듭니다.

**통과 증거:** `course_outputs/receipt_result.xlsx`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 실행 모드를 먼저 확인합니다.

- `LIVE`: 현재 파일에 실제 모델을 실행한 결과
- `PREPARED_FALLBACK`: 공개 샘플을 사람이 검수해 둔 복구 결과
- 3분 이상 멈추면 실행을 중지하고 복구 결과로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


In [ ]:
import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_PREPARED") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_PREPARED_INPUT=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())


In [ ]:
import importlib.util
import subprocess
if importlib.util.find_spec("openpyxl") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openpyxl==3.1.5"]
    )
from copy import deepcopy
from datetime import date
from openpyxl import Workbook, load_workbook


In [ ]:
GOLDEN_OCR_TEXT = '이태리집\n거래일시 2025-10-04 12:33:37\n페퍼로니 앤 치즈 29,000 1 29,000\n토마토 파스타 14,000 1 14,000\n수제 돈가스 13,000 1 13,000\n새우 칠리치 필라 14,000 1 14,000\n콜라 2,000 3 6,000\n합계 금액 76,000\n부가세 과세물품가액 69,094\n부가세 6,906\n'
GOLDEN_RECEIPT = {'document_type': 'receipt', 'store_name': '이태리집', 'date': '2025-10-04', 'total_amount': 76000, 'items': [{'name': '페퍼로니 앤 치즈', 'quantity': 1, 'unit_price': 29000, 'line_total': 29000}, {'name': '토마토 파스타', 'quantity': 1, 'unit_price': 14000, 'line_total': 14000}, {'name': '수제 돈가스', 'quantity': 1, 'unit_price': 13000, 'line_total': 13000}, {'name': '새우 칠리치 필라', 'quantity': 1, 'unit_price': 14000, 'line_total': 14000}, {'name': '콜라', 'quantity': 3, 'unit_price': 2000, 'line_total': 6000}], 'adjustments': {'discount': 0, 'tax': 0, 'service': 0, 'rounding': 0}, 'tax_breakdown': {'mode': 'included_in_item_prices', 'supply_amount': 69094, 'vat': 6906, 'payable_total': 76000}, 'raw_values': {'store_name': '이태리집', 'date': '2025-10-04 12:33:37', 'total_amount': '76,000'}, 'cleaned_values': {'store_name': '이태리집', 'date': '2025-10-04', 'total_amount': 76000}, 'evidence': {'store_name': {'raw_value': '이태리집', 'line': 1}, 'date': {'raw_value': '거래일시 2025-10-04 12:33:37', 'line': 2}, 'total_amount': {'raw_value': '합계 금액 76,000', 'line': 8}}, 'source_mode': 'prepared_fixture_rule_extraction'}


In [ ]:
input_path = OUTPUT_DIR / "receipt.json"
USE_PREPARED_INPUT = VALIDATION_MODE
if not input_path.exists() and not USE_PREPARED_INPUT:
    upload_previous_artifact("receipt.json")
if input_path.exists():
    receipt = json.loads(input_path.read_text(encoding="utf-8"))
    INPUT_MODE = "PREVIOUS_LESSON"
else:
    receipt = deepcopy(GOLDEN_RECEIPT)
    INPUT_MODE = "PREPARED_FALLBACK"


def validate_receipt(data):
    warnings, errors = [], []
    for field in ("store_name", "date", "total_amount", "items"):
        if data.get(field) in (None, "", []):
            errors.append(f"필수값 누락: {field}")
    try:
        parsed_date = date.fromisoformat(data.get("date", ""))
        if parsed_date > date.today():
            warnings.append("미래 날짜입니다. 원본을 확인하세요.")
    except ValueError:
        errors.append("date는 YYYY-MM-DD 형식이어야 합니다.")

    total = data.get("total_amount")
    if isinstance(total, bool) or not isinstance(total, int) or total < 0:
        errors.append("total_amount는 0 이상의 정수여야 합니다.")
    item_sum = 0
    for index, item in enumerate(data.get("items") or [], start=1):
        values = [item.get(key) for key in ("quantity", "unit_price", "line_total")]
        if not all(isinstance(value, int) and not isinstance(value, bool) for value in values):
            errors.append(f"{index}번째 품목 금액 형식 오류")
            continue
        if values[0] * values[1] != values[2]:
            errors.append(f"{index}번째 품목 수량×단가 오류")
        item_sum += values[2]
    adjustments = data.get("adjustments") or {}
    expected = (
        item_sum
        - adjustments.get("discount", 0)
        + adjustments.get("tax", 0)
        + adjustments.get("service", 0)
        + adjustments.get("rounding", 0)
    )
    if isinstance(total, int) and not isinstance(total, bool) and expected != total:
        errors.append(f"품목·조정 후 합계 {expected:,}원과 총액 {total:,}원이 다릅니다.")
    tax_breakdown = data.get("tax_breakdown")
    if tax_breakdown and tax_breakdown.get("mode") == "included_in_item_prices":
        supply = tax_breakdown.get("supply_amount")
        vat = tax_breakdown.get("vat")
        payable = tax_breakdown.get("payable_total")
        if not all(isinstance(value, int) and not isinstance(value, bool)
                   for value in (supply, vat, payable)):
            errors.append("포함세액 내역은 정수 금액이어야 합니다.")
        elif supply + vat != payable or payable != total:
            errors.append("공급가액·포함 부가세·총액 관계가 맞지 않습니다.")
        if adjustments.get("tax", 0) != 0:
            errors.append("포함 부가세를 adjustments.tax에 다시 더하면 이중 계산됩니다.")
    for field in ("store_name", "date", "total_amount"):
        if not (data.get("evidence") or {}).get(field):
            warnings.append(f"{field}의 원본 근거가 없습니다.")
    return {"valid": not errors, "warnings": warnings, "errors": errors}


validation = validate_receipt(receipt)
assert validation["valid"], validation
print("입력 모드:", INPUT_MODE)
print("검증:", validation)


In [ ]:
def safe_text(value):
    if isinstance(value, str) and value.lstrip(" \t\r\n").startswith(
        ("=", "+", "-", "@")
    ):
        return "'" + value
    return value


def save_reviewed_excel(data, validation, review_record, output_path, source_text):
    if not validation["valid"]:
        return False
    if review_record.get("decision") not in {"APPROVED", "CHANGED"}:
        return False

    workbook = Workbook()
    summary = workbook.active
    summary.title = "검토_요약"
    summary.append([
        "field", "raw_value", "cleaned_value", "final_value",
        "decision", "reviewer", "reviewed_at", "change_reason",
    ])
    raw = data.get("raw_values") or {}
    cleaned = data.get("cleaned_values") or {}
    for field in ("store_name", "date", "total_amount"):
        summary.append([
            field,
            safe_text(raw.get(field)),
            safe_text(cleaned.get(field)),
            safe_text(data.get(field)),
            review_record["decision"],
            safe_text(review_record["reviewer"]),
            review_record["reviewed_at"],
            safe_text(review_record["note"]),
        ])

    items = workbook.create_sheet("품목")
    items.append(["품목", "수량", "단가", "금액"])
    for item in data["items"]:
        items.append([
            safe_text(item["name"]),
            item["quantity"],
            item["unit_price"],
            item["line_total"],
        ])

    evidence = workbook.create_sheet("원문_근거")
    evidence.append(["source_mode", data.get("source_mode")])
    evidence.append(["ocr_text", safe_text(source_text)])
    evidence.append(["evidence", safe_text(json.dumps(
        data.get("evidence") or {}, ensure_ascii=False
    ))])
    workbook.save(output_path)
    return True


## 시나리오 A. 기본값은 차단

사람이 원본을 보기 전에는 결과가 유효해도 다운로드를 열지 않습니다.


In [ ]:
blocked_path = OUTPUT_DIR / "pending_review.xlsx"
PENDING_REVIEW = {
    "decision": "PENDING",
    "reviewer": "",
    "reviewed_at": "",
    "note": "",
}
assert not save_reviewed_excel(
    receipt, validation, PENDING_REVIEW, blocked_path, GOLDEN_OCR_TEXT
)
assert not blocked_path.exists()
print("DEFAULT_BLOCKED PASS: 미승인 Excel 없음")


## 시나리오 B. 전체 정답 공개 — 원본 확인 후 실행

아래 셀은 승인 기록의 **완성 정답**입니다. 원본의 상호명·날짜·품목·총액을
직접 대조한 뒤 실행합니다. 결정·검토자·시각·메모가 Excel에 남습니다.


In [ ]:
REVIEW_RECORD = {
    "decision": "APPROVED",
    "reviewer": "learner",
    "reviewed_at": "2026-07-28T15:30:00+09:00",
    "note": "공개 비식별 원본과 상호명·날짜·품목·총액 대조 완료",
}
output_path = OUTPUT_DIR / "receipt_result.xlsx"
assert save_reviewed_excel(
    receipt, validation, REVIEW_RECORD, output_path, GOLDEN_OCR_TEXT
)
saved = load_workbook(output_path)
assert saved.sheetnames == ["검토_요약", "품목", "원문_근거"]
assert saved["검토_요약"]["E2"].value == "APPROVED"
print("REVIEWED_APPROVED PASS:", output_path, saved.sheetnames)
print("CHECKPOINT 1/1 PASS: 미승인 차단 + 승인 후 Excel")
download_artifact(output_path)
